# Customer Churn Prediction: EDA and Baseline Model

This notebook walks through the first version of the customer churn project.

Goal: predict whether a telecom customer will churn and identify business factors associated with churn.

## 1. Load Data

The cleaned dataset is created by running `python src/clean_data.py` from the project root.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data_path = Path('../data/processed/telco_churn_clean.csv')
if not data_path.exists():
    data_path = Path('data/processed/telco_churn_clean.csv')

df = pd.read_csv(data_path)
df.head()

In [ ]:
df.shape, df['churn'].mean()

## 2. Churn Rate by Business Features

We first compare churn rates across features that a business can act on: contract type, internet service, and payment method.

In [ ]:
def churn_rate_by(column):
    return (
        df.groupby(column)['churn']
        .agg(customers='count', churn_rate='mean')
        .sort_values('churn_rate', ascending=False)
    )

churn_rate_by('contract')

In [ ]:
churn_rate_by('paymentmethod')

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=df, x='contract', y='churn', errorbar=None)
plt.title('Churn Rate by Contract Type')
plt.ylabel('Churn rate')
plt.xlabel('Contract type')
plt.show()

## 3. Baseline Modeling

For churn prediction, recall for the churn class matters because the business wants to find customers at risk before they leave.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['churn'])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ]
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, predictions))
print('ROC-AUC:', roc_auc_score(y_test, probabilities))

## 4. Business Interpretation

Early findings:

- Month-to-month contracts have much higher churn than one-year or two-year contracts.
- Electronic check users have the highest churn rate among payment methods.
- Logistic Regression gives strong churn recall, which is useful when the business wants to identify at-risk customers for retention campaigns.